Setting parameters

In [36]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta
from IPython.display import display, clear_output

# ==========================================
# 1. PARAMETERS & PATHS
# ==========================================

# Base paths
BASE_DIR = r"W:\Dzial Business process\Dzial Analiz\CENNIKI"
PATH_DIRECTORY = r"C:\DanePYTHON\EstEst_evaluation\directory.xlsx"
PATH_ACTUALS = r"C:\DanePYTHON\EstEst_evaluation\Time est_est_evaluaion shr monthly.csv"

# Timeframe
START_YEAR = 2025
CURRENT_DATE = datetime.now()
END_YEAR = CURRENT_DATE.year + 1 

# Column Configurations
EMPLOYEES = ['DT', 'MJ', 'AKB', 'BW', 'KB', 'BD', 'AK', 'RD', 'PW', 'WK', 'ŁŁ']
EXCLUDED_CHANNELS = ['ATV', 'SHR', 'Dynamic other', 'TEMATYKI', 'TVN', 'TVP1', 'TVP2']
# Defining the columns we want to extract from the raw files
TARGET_COLUMNS = EMPLOYEES + ['STACJA', 'Czyja', 'Wynik SHR']

Data importing and processing

In [37]:
raw_data_frames = []
print("Starting data extraction...")

for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        mm = f"{month:02d}"
        yyyy = str(year)
        
        folder_path = os.path.join(BASE_DIR, f"{yyyy} cenniki", f"{yyyy}_{mm}")
        file_name = f"Est_est_{yyyy}_{mm}_robocze.xlsx"
        full_path = os.path.join(folder_path, file_name)
        
        if os.path.exists(full_path):
            try:
                # 1. Wczytujemy TYLKO te kolumny, które fizycznie istnieją w tym jednym pliku
                df_temp = pd.read_excel(full_path, usecols=lambda x: x in TARGET_COLUMNS)
                
                # 2. Od razu zmieniamy nazwy i czyścimy ten jeden miesiąc
                df_temp.rename(columns={'STACJA': 'Channel_Provys', 'Czyja': 'owner', 'Wynik SHR': 'avg_estSHR'}, inplace=True)
                df_temp['Channel_Provys'] = df_temp['Channel_Provys'].str.strip()
                df_temp['Channel_Provys'].replace('', pd.NA, inplace=True)
                
                # Filtrowanie stacji
                df_temp = df_temp[
                    (~df_temp['Channel_Provys'].isin(EXCLUDED_CHANNELS)) & 
                    (df_temp['Channel_Provys'].notna())
                ]
                
                # 3. Sprawdzamy, którzy PRACOWNICY SĄ FAKTYCZNIE W TYM PLIKU (jeśli ŁŁ nie ma w lipcu, tu odpadnie)
                emp_in_file = [emp for emp in EMPLOYEES if emp in df_temp.columns]
                
                df_temp['Year'] = int(year)
                df_temp['Month'] = int(month)
                
                # 4. Meltujemy (spłaszczamy) tylko dane z tego miesiąca dla obecnych pracowników
                df_melted = df_temp.melt(
                    id_vars=['Year', 'Month', 'Channel_Provys', 'owner', 'avg_estSHR'], 
                    value_vars=emp_in_file,
                    var_name='Employee', 
                    value_name='Estimation'
                )
                
                raw_data_frames.append(df_melted)
            except Exception as e:
                print(f"Error reading {file_name}: {e}")

# 5. Dopiero teraz łączymy wszystkie pliki w jedną długą tabelę
df_estest = pd.concat(raw_data_frames, ignore_index=True)

# 6. Konwersja na liczby 
# (Zostawiam * 100, jeśli tak było potrzebne dla cenników, jeśli nie - usuń * 100)
df_estest['Estimation'] = (pd.to_numeric(df_estest['Estimation'], errors='coerce') * 100).round(2)
df_estest['avg_estSHR'] = (pd.to_numeric(df_estest['avg_estSHR'], errors='coerce') * 100).round(2)

# 7. FILTRY BEZPIECZEŃSTWA: wyrzucamy puste komórki oraz fałszywe ZERA
df_estest.dropna(subset=['Estimation'], inplace=True)
df_estest = df_estest[df_estest['Estimation'] != 0]

print(f"✅ Successfully created df_estest with {len(df_estest)} rows.")
df_estest.head(3)

Starting data extraction...


C:\Users\rafal.daszuta\AppData\Local\Temp\ipykernel_16752\1868425481.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_temp['Channel_Provys'].replace('', pd.NA, inplace=True)
C:\Users\rafal.daszuta\AppData\Local\Temp\ipykernel_16752\1868425481.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

✅ Successfully created df_estest with 17995 rows.


,Year,Month,Channel_Provys,owner,avg_estSHR,Employee,Estimation
0,2025,1,Polsat,Maciej,7.25,DT,7.14
1,2025,1,Fokus TV,Adam,0.93,DT,0.86
2,2025,1,Nowa TV,Adam,0.45,DT,0.44


Channel translation and joining actual values

In [38]:
# ==========================================
# 3. MAP CHANNELS & JOIN ACTUAL SHR
# ==========================================

# 3A. Map Provys to TechEdge
df_dict = pd.read_excel(PATH_DIRECTORY, usecols=['Channel_Provys', 'Channel_TechEdge'])
df_dict['Channel_Provys'] = df_dict['Channel_Provys'].str.strip()
df_estest = df_estest.merge(df_dict, on='Channel_Provys', how='left')

missing_mapping = df_estest[df_estest['Channel_TechEdge'].isna()]
if not missing_mapping.empty:
    print("⚠️ Missing Channel Translations found! E.g.:", missing_mapping['Channel_Provys'].unique()[:5])

# ==========================================
# 3B. Import and clean Actuals
# ==========================================

# Dodanie parametru decimal=',' rozwiązuje problem już na etapie wczytywania pliku
df_actuals = pd.read_csv(PATH_ACTUALS, skiprows=10, decimal=',')
df_actuals.rename(columns={'Channel': 'Channel_TechEdge', 'Month in Year': 'Month', 'Share': 'actual_SHR'}, inplace=True)

df_actuals['Year'] = pd.to_numeric(df_actuals['Year'], errors='coerce')
df_actuals['Month'] = pd.to_numeric(df_actuals['Month'], errors='coerce')

# Zabezpieczenie: zamiana przecinków na kropki na wypadek, gdyby Pandas wczytał to jako string
df_actuals['actual_SHR'] = df_actuals['actual_SHR'].astype(str).str.replace(',', '.')

# Convert Actual SHR to numeric -> multiply by 100 -> round to 2 decimals
df_actuals['actual_SHR'] = (pd.to_numeric(df_actuals['actual_SHR'], errors='coerce')).round(2)

df_actuals.dropna(subset=['Channel_TechEdge', 'Year', 'Month'], inplace=True)

# 3C. Merge Actuals to main dataframe
df_estest = df_estest.merge(df_actuals[['Channel_TechEdge', 'Year', 'Month', 'actual_SHR']], on=['Channel_TechEdge', 'Year', 'Month'], how='left')

missing_actuals = df_estest[df_estest['actual_SHR'].isna() & df_estest['Channel_TechEdge'].notna()]
if not missing_actuals.empty:
    print("⚠️ WARNING: Some mapped channels are missing Actual SHR Data.")

print("✅ Data mapping and merging complete.")

⚠️ WARNING: Some mapped channels are missing Actual SHR Data.
✅ Data mapping and merging complete.


RMSE

In [39]:
current_month_start = CURRENT_DATE.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
start_date = current_month_start - relativedelta(months=12)
df_estest['date_col'] = pd.to_datetime(df_estest[['Year', 'Month']].assign(day=1))
df_subset = df_estest[(df_estest['date_col'] >= start_date) & (df_estest['date_col'] < current_month_start)].copy()
df_subset = df_subset.dropna(subset=['Estimation', 'actual_SHR'])

# RMSE Calculation
df_subset['sq_error'] = (df_subset['Estimation'] - df_subset['actual_SHR'])**2
df_rmse = df_subset.groupby(['Year', 'Month', 'Employee'])['sq_error'].mean().reset_index()
df_rmse['RMSE'] = np.sqrt(df_rmse['sq_error'])

# Pivot
df_rmse['Period'] = df_rmse['Year'].astype(str) + "-" + df_rmse['Month'].astype(str).str.zfill(2)
pivot_df = df_rmse.pivot(index='Employee', columns='Period', values='RMSE')

if not pivot_df.empty:
    # SORTOWANIE: najpierw względem ostatniego miesiąca
    latest_period = pivot_df.columns.max()
    pivot_df = pivot_df.sort_values(by=latest_period, ascending=True)

    # NOWOŚĆ: Wymuszenie, by wszyscy pracownicy z listy zawsze byli w tabeli (nawet jeśli mają wszędzie braki)
    # Tylko ci, którzy są na oficjalnej liście EMPLOYEES
    pivot_df = pivot_df.reindex(EMPLOYEES)
    
    # Przesunięcie osób z samymi brakami na dół tabeli
    # (reindex może zmienić kolejność, więc robimy szybkie ponowne posortowanie)
    pivot_df = pivot_df.sort_values(by=latest_period, ascending=True, na_position='last')

    def highlight_top3(col):
        styles = [''] * len(col)
        # na_option='bottom' sprawia, że braki (NaN) nie zaburzają rankingu (nie dostają 1, 2, 3)
        ranks = col.rank(method='min', ascending=True, na_option='bottom')
        for i, rank in enumerate(ranks):
            if pd.isna(rank): continue # Pomija kolorowanie dla braków
            if rank == 1: styles[i] = 'background-color: #74c476; color: black;'
            elif rank == 2: styles[i] = 'background-color: #a1d99b; color: black;'
            elif rank == 3: styles[i] = 'background-color: #c7e9c0; color: black;'
        return styles

    print(f"📊 BŁĄD RMSE PRACOWNIKÓW (W punktach procentowych)")
    # na_rep="-" automatycznie zamieni wszystkie wartości NaN (w tym brak ŁŁ w 07.26) na "-"
    display(pivot_df.style.apply(highlight_top3, axis=0).format("{:.2f}", na_rep="-"))
else:
    print("Brak danych z ostatnich 12 miesięcy do wygenerowania tabeli pracowniczej.")

📊 BŁĄD RMSE PRACOWNIKÓW (W punktach procentowych)


Period,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07
Employee,,,,,,,,,,,,
KB,0.09,-,0.10,0.07,0.15,0.09,0.12,0.07,0.09,0.05,0.07,0.08
BD,-,-,0.08,0.09,0.14,-,-,0.06,0.08,0.05,0.08,0.09
BW,0.09,0.10,0.11,0.08,0.14,-,-,0.07,0.09,0.05,0.08,0.10
PW,0.09,0.08,0.09,-,0.13,0.09,0.11,-,0.07,0.06,0.09,0.10
WK,0.09,0.09,0.08,0.08,0.15,0.07,0.10,-,0.08,0.05,0.07,0.10
AK,0.09,-,0.12,0.10,0.16,0.10,0.10,0.08,0.08,0.06,0.10,0.11
DT,0.09,0.11,0.09,0.08,0.13,0.08,0.13,0.08,0.07,0.05,0.08,0.11
RD,0.13,0.10,0.09,0.09,0.13,0.06,0.08,0.06,0.07,0.05,0.09,0.11
MJ,0.09,0.11,0.11,0.07,0.16,0.11,0.11,0.07,0.09,0.06,0.08,0.11


Błędy na stacjach

In [40]:
# 1. Upewnij się, że masz kolumnę z kwadratami różnic (sq_error)
df_subset['sq_error'] = (df_subset['Estimation'] - df_subset['actual_SHR'])**2

# 2. Zamiast .mean(), policz sumę i podziel przez liczbę elementów
# Używamy len(df_subset), co jest odpowiednikiem ILE.NIEPUSTYCH po wykonaniu dropna()
sum_sq_errors = df_subset['sq_error'].sum()
count = len(df_subset)

rmse_excel_style = np.sqrt(sum_sq_errors / count)

print(f"Suma kwadratów: {sum_sq_errors}")
print(f"Liczba obserwacji (ILE.NIEPUSTYCH): {count}")
print(f"Wynik RMSE: {rmse_excel_style}")

Suma kwadratów: 102.45779999999998
Liczba obserwacji (ILE.NIEPUSTYCH): 11521
Wynik RMSE: 0.09430341603854281


In [43]:
# Target month is precisely 1 month before CURRENT_DATE
target_date = CURRENT_DATE - relativedelta(months=1)
target_year, target_month = target_date.year, target_date.month

df_latest = df_estest[(df_estest['Year'] == target_year) & (df_estest['Month'] == target_month)].copy()

if df_latest.empty:
    print(f"⚠️ Brak danych dla {target_year}-{target_month:02d}.")
else:
    df_channel = df_latest.drop_duplicates(subset=['Channel_TechEdge']).copy()
    df_channel = df_channel.dropna(subset=['avg_estSHR', 'actual_SHR'])
    
    # Total Error = Różnica w Punktach Procentowych (np. 3.00 - 2.50 = 0.50 pp)
    df_channel['Total_Error'] = df_channel['actual_SHR'] - df_channel['avg_estSHR']
    df_channel['Abs_Total_Error'] = df_channel['Total_Error'].abs()
    
    # Pct Error = Relatywny błąd procentowy (np. 0.50 / 2.50 = 20%)
    df_channel['Pct_Error'] = df_channel['Total_Error'] / df_channel['actual_SHR']
    df_channel['Abs_Pct_Error'] = df_channel['Pct_Error'].abs()
    
    show_cols = ['Channel_TechEdge', 'avg_estSHR', 'actual_SHR', 'Total_Error', 'Pct_Error']
    
    format_dict = {
        'avg_estSHR': "{:.2f}", 
        'actual_SHR': "{:.2f}",
        'Total_Error': "{:+.2f}", # Znak +/- dla punktów procentowych
        'Pct_Error': "{:+.2%}"    # Standardowe formatowanie procentowe z symbolem %
    }
    
    # TABLE 1: Total Error
    print(f"🔴 TOP 15: NAJWIĘKSZE BŁĘDY CAŁKOWITE (w pkt proc.) DLA {target_year}-{target_month:02d}")
    df_total_error = df_channel.sort_values(by='Abs_Total_Error', ascending=False)[show_cols].head(15)
    
    max_val_total = df_total_error['Total_Error'].abs().max() if not df_total_error.empty else 1.0
    display(df_total_error.style.format(format_dict)
            .background_gradient(subset=['Total_Error'], cmap='coolwarm', vmin=-max_val_total, vmax=max_val_total)
            .hide(axis="index"))
    
    print("\n" + "="*80 + "\n")
    
   # TABLE 2: Percentage Error
    print(f"🔴 TOP 15: NAJWIĘKSZE BŁĘDY WZGLĘDNE (procentowo od bazy) DLA {target_year}-{target_month:02d} (tylko SHR > 0.05)")
    
    # ZASTOSOWANY FILTR: Wybieramy tylko te wiersze, gdzie actual_SHR jest większe niż 0.05
    df_pct_error = df_channel[df_channel['actual_SHR'] > 0.05].sort_values(by='Abs_Pct_Error', ascending=False)[show_cols].head(15)
    
    max_val_pct = df_pct_error['Pct_Error'].abs().max() if not df_pct_error.empty else 1.0
    display(df_pct_error.style.format(format_dict)
            .background_gradient(subset=['Pct_Error'], cmap='coolwarm', vmin=-max_val_pct, vmax=max_val_pct)
            .hide(axis="index"))

🔴 TOP 15: NAJWIĘKSZE BŁĘDY CAŁKOWITE (w pkt proc.) DLA 2026-07


Channel_TechEdge,avg_estSHR,actual_SHR,Total_Error,Pct_Error
Polsat,6.85,6.15,-0.70,-11.38%
TV PULS,3.48,3.82,+0.34,+8.90%
Republika,3.19,2.90,-0.29,-10.00%
Wydarzenia24,1.01,1.28,+0.27,+21.09%
Polsat Sport 1,0.93,0.70,-0.23,-32.86%
Novelas,0.10,0.23,+0.13,+56.52%
Puls 2,1.55,1.68,+0.13,+7.74%
AXN Black,0.11,0.22,+0.11,+50.00%
History,0.41,0.30,-0.11,-36.67%
Polsat Viasat Explore,0.08,0.17,+0.09,+52.94%




🔴 TOP 15: NAJWIĘKSZE BŁĘDY WZGLĘDNE (procentowo od bazy) DLA 2026-07 (tylko SHR > 0.05)


Channel_TechEdge,avg_estSHR,actual_SHR,Total_Error,Pct_Error
Canal+ SPORT5,0.16,0.10,-0.06,-60.00%
Novelas,0.10,0.23,+0.13,+56.52%
4FUN DANCE,0.05,0.11,+0.06,+54.55%
Polsat Viasat Explore,0.08,0.17,+0.09,+52.94%
AXN Black,0.11,0.22,+0.11,+50.00%
Polsat X,0.03,0.06,+0.03,+50.00%
4FUN KIDS,0.09,0.06,-0.03,-50.00%
Planete+,0.15,0.10,-0.05,-50.00%
BBC First,0.22,0.16,-0.06,-37.50%
History,0.41,0.30,-0.11,-36.67%
